# Notebook 03: central fusion model on raw multi-rate contract

Этот notebook продолжает цепочку `01 -> 02 -> 03`: первые два notebook описали сырые данные и собрали воспроизводимый multi-rate contract, а здесь поверх этого contract строится центральная мультимодальная модель. Текстовая ветка использует HF encoder `google/embeddinggemma-300m`, как в фактической реализации старого notebook, но без legacy data fallback и без вывода секретов.

### Локальный вывод
`03` не меняет target policy из `02` и не пересобирает окна. Его задача — проверить, что над существующим contract можно воспроизводимо собрать missing-aware fusion model и подготовить честный протокол полного обучения без leakage.


## 1. From baseline contract to central fusion model

| Fact from `02` | Design decision | Risk controlled |
|---|---|---|
| strict `run_id = workload/scenario` | group-aware train/val/test | leakage between windows from one run |
| 10s micro / 60s macro | separate log_micro/log_macro branches | loss of multi-rate meaning |
| 2,160 KPI columns | tensorization with series order + optional cap | high-dimensional flat overfit |
| `has_both_modalities` near 0.568 | missing-aware gate + masks | broken fusion when one modality is absent |
| availability-only AP is non-trivial | compare with shortcut baseline | coverage shortcut |
| proxy anchors are common | anchor-policy slices | over-interpretation of quality |
| HF text encoder is external to supervised training | cache embeddings and store encoder manifest | reproducibility of text branch |

Центральная модель здесь одна: log macro branch, log micro branch, KPI branch, gated late fusion и две heads для `y_warning`/`y_risk`. Baseline controls используются только для сравнения в full-training режиме.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import random
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

try:
    import joblib
except Exception:
    joblib = None

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path.cwd()
CONTRACT_PATH = ROOT / 'outputs' / 'raw_multirate' / 'aligned_multirate_windows.csv'
SCHEMA_PATH = ROOT / 'outputs' / 'raw_multirate' / 'feature_schema.json'
BASELINE_METRICS_PATH = ROOT / 'outputs' / 'raw_multirate' / 'baseline_summary_metrics.json'
BASELINE_PRED_PATH = ROOT / 'outputs' / 'raw_multirate' / 'model_predictions_smoke.csv'
SPLIT_REPORT_PATH = ROOT / 'outputs' / 'raw_multirate' / 'split_report.csv'
OUT_ROOT = ROOT / 'outputs' / 'raw_central_model'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_FULL_TRAINING = False
SEED = 42
USE_HF_GOOGLE_EMBEDDER = True
HF_GOOGLE_EMBEDDER_MODEL = 'google/embeddinggemma-300m'
ALLOW_HF_DOWNLOAD = True
HF_EMBED_BATCH_SIZE = 64
HASHING_FALLBACK_DIM = 192
CACHE_DIR = OUT_ROOT / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HIDDEN_DIM = 64
BATCH_SIZE = 128
DRY_RUN_BATCH_SIZE = 64
MAX_EPOCHS = 120
PATIENCE = 14
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MODALITY_DROPOUT = 0.20
LAMBDA_RISK = 0.35
LAMBDA_AUX = 0.12
USE_AUX_HEADS = True

STAT_ORDER = ['mean', 'last', 'std', 'delta', 'slope']
LOG_MACRO_COLS = ['macro_log_count', 'macro_unique_components', 'macro_warn_count', 'macro_error_count', 'macro_keyword_count', 'macro_log_rate']
LOG_MICRO_COLS = ['micro_log_count', 'micro_unique_components', 'micro_warn_count', 'micro_error_count', 'micro_keyword_count', 'micro_log_rate', 'micro_error_rate']
MASK_COLS = ['log_available', 'kpi_available', 'has_both_modalities', 'kpi_avail_ratio', 'kpi_series_available', 'kpi_feature_count', 'kpi_recency_sec']
MODEL_MASK_COLS = ['log_available', 'kpi_available', 'has_both_modalities', 'kpi_avail_ratio']
KPI_MICRO_COLS = ['kpi_series_available', 'kpi_feature_count', 'kpi_avail_ratio', 'kpi_recency_sec']
TARGET_COLS = ['y_warning', 'y_risk']

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def json_sanitize(obj):
    if isinstance(obj, dict):
        return {str(k): json_sanitize(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_sanitize(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return val if math.isfinite(val) else None
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if obj is pd.NA:
        return None
    return obj

def write_json_safe(path, payload):
    cleaned = json_sanitize(payload)
    with Path(path).open('w', encoding='utf-8') as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2, allow_nan=False)

def finite_float(x):
    try:
        val = float(x)
    except Exception:
        return None
    return val if math.isfinite(val) else None

set_seed(SEED)
print({'RUN_FULL_TRAINING': RUN_FULL_TRAINING, 'encoder': HF_GOOGLE_EMBEDDER_MODEL, 'output_root': str(OUT_ROOT.relative_to(ROOT))})


{'RUN_FULL_TRAINING': False, 'encoder': 'google/embeddinggemma-300m', 'output_root': 'outputs\\raw_central_model'}


## 2. Strict contract load and validation

Этот блок читает только artifacts из `outputs/raw_multirate/`. Если contract или schema отсутствуют, notebook падает сразу: silent fallback на старые результаты запрещён.


In [2]:
def require_file(path, label):
    if not Path(path).exists():
        raise FileNotFoundError(f'Missing {path}. Run notebook 02 first before running 03: {label}')

for path, label in [
    (CONTRACT_PATH, 'aligned multi-rate contract'),
    (SCHEMA_PATH, 'feature schema'),
    (BASELINE_METRICS_PATH, 'baseline metrics'),
]:
    require_file(path, label)

with SCHEMA_PATH.open('r', encoding='utf-8') as f:
    schema = json.load(f)
with BASELINE_METRICS_PATH.open('r', encoding='utf-8') as f:
    baseline_summary = json.load(f)

df = pd.read_csv(CONTRACT_PATH, low_memory=False)
df['log_text'] = df['log_text'].fillna('').astype(str)

base_required = [
    'workload', 'scenario', 'run_id', 'group_id', 'failure_family', 'scenario_type',
    'window_ts', 'run_start', 'run_end', 'event_anchor', 'anchor_policy', 'minutes_before_fail',
    'y_warning', 'y_risk', 'y_anomaly', 'target_available', 'target_policy', 'log_text',
    'log_available', 'kpi_available', 'kpi_series_available', 'kpi_feature_count',
    'kpi_avail_ratio', 'kpi_recency_sec', 'has_both_modalities', 'hard_negative',
    'parsed_label_available', 'strict_path_pair', 'micro_window_sec', 'macro_window_sec'
]
required_cols = base_required + LOG_MACRO_COLS + LOG_MICRO_COLS
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required 03 contract columns: {missing_required[:20]}')

split_fallback_used = False
split_fallback_reason = None
if 'split' not in df.columns:
    if not SPLIT_REPORT_PATH.exists():
        raise ValueError('Column split is absent and split_report.csv is unavailable for reconstruction')
    split_report_df = pd.read_csv(SPLIT_REPORT_PATH)
    test_row = split_report_df[(split_report_df['section'] == 'workloads') & (split_report_df['split'] == 'test')]
    if test_row.empty:
        raise ValueError('Column split is absent and test workloads cannot be recovered from split_report.csv')
    raw_groups = test_row['groups'].iloc[0]
    test_workloads = set(json.loads(raw_groups))
    df['split'] = np.where(df['workload'].isin(test_workloads), 'test', 'train')
    df['split_group'] = df['workload']
    split_fallback_used = True
    split_fallback_reason = 'split column reconstructed from split_report.csv workload holdout'
else:
    df['split'] = df['split'].astype(str).str.lower()

if 'split_group' not in df.columns:
    df['split_group'] = df['workload']

duplicate_key_count = int(df.duplicated(['run_id', 'window_ts', 'micro_window_sec']).sum())
if duplicate_key_count != 0:
    raise ValueError(f'Duplicate (run_id, window_ts, micro_window_sec) rows detected: {duplicate_key_count}')

integrity_cols = TARGET_COLS + ['y_anomaly'] + [c for c in MASK_COLS if c != 'kpi_recency_sec']
integrity_report = {}
for col in integrity_cols:
    x = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    integrity_report[col] = {'missing_or_inf': int(x.isna().sum()), 'min': finite_float(x.min()), 'max': finite_float(x.max())}
recency_raw = pd.to_numeric(df['kpi_recency_sec'], errors='coerce').replace([np.inf, -np.inf], np.nan)
recency_missing_count = int(recency_raw.isna().sum())
if recency_missing_count:
    recency_fill_value = float(recency_raw.dropna().max()) if recency_raw.notna().any() else 0.0
    df['kpi_recency_sec'] = recency_raw.fillna(recency_fill_value).astype(float)
else:
    recency_fill_value = None
    df['kpi_recency_sec'] = recency_raw.astype(float)
integrity_report['kpi_recency_sec'] = {'missing_or_inf_before_imputation': recency_missing_count, 'imputed_fill_value': recency_fill_value, 'min_after': finite_float(df['kpi_recency_sec'].min()), 'max_after': finite_float(df['kpi_recency_sec'].max())}
bad_integrity = {k: v for k, v in integrity_report.items() if v.get('missing_or_inf', 0) > 0}
if bad_integrity:
    raise ValueError(f'Target/mask columns contain missing or infinite values: {bad_integrity}')

contract_summary = {
    'rows': int(len(df)),
    'columns': int(df.shape[1]),
    'runs': int(df['run_id'].nunique()),
    'workloads': int(df['workload'].nunique()),
    'duplicate_key_count': duplicate_key_count,
    'y_warning_prevalence': float(df['y_warning'].mean()),
    'y_risk_prevalence': float(df['y_risk'].mean()),
    'kpi_available_share': float(df['kpi_available'].mean()),
    'has_both_modalities_share': float(df['has_both_modalities'].mean()),
    'kpi_recency_missing_imputed': recency_missing_count,
    'split_fallback_used': split_fallback_used,
    'split_fallback_reason': split_fallback_reason,
}
split_summary = (df.groupby('split')
    .agg(rows=('run_id', 'size'), runs=('run_id', 'nunique'), workloads=('workload', 'nunique'),
         warning_prevalence=('y_warning', 'mean'), risk_prevalence=('y_risk', 'mean'),
         both_modalities_share=('has_both_modalities', 'mean'))
    .reset_index())
class_presence = []
for split_name, part in df.groupby('split'):
    for target in TARGET_COLS:
        counts = part[target].value_counts().to_dict()
        class_presence.append({'split': split_name, 'target': target, 'negatives': int(counts.get(0, 0)), 'positives': int(counts.get(1, 0)), 'both_classes': part[target].nunique() == 2})
class_presence_df = pd.DataFrame(class_presence)
required_class_checks = class_presence_df[class_presence_df['split'].isin(['train', 'test'])]
if not bool(required_class_checks['both_classes'].all()):
    raise ValueError('Train/test split does not contain both classes for all targets')

display(pd.DataFrame([contract_summary]))
display(split_summary)
display(class_presence_df)

print('Contract loaded and validated from:', CONTRACT_PATH.relative_to(ROOT))


,rows,columns,runs,workloads,duplicate_key_count,y_warning_prevalence,y_risk_prevalence,kpi_available_share,has_both_modalities_share,kpi_recency_missing_imputed,split_fallback_used,split_fallback_reason
0,10237,2211,440,16,0,0.292762,0.122595,0.999609,0.56794,4,False,None


,split,rows,runs,workloads,warning_prevalence,risk_prevalence,both_modalities_share
0,test,2567,113,4,0.283600,0.128944,0.472926
1,train,7670,327,12,0.295828,0.120469,0.599739


,split,target,negatives,positives,both_classes
0,test,y_warning,1839,728,True
1,test,y_risk,2236,331,True
2,train,y_warning,5401,2269,True
3,train,y_risk,6746,924,True


Contract loaded and validated from: outputs\raw_multirate\aligned_multirate_windows.csv


### Локальный вывод
Contract из `02` используется как единственный источник supervised windows. Проверки выше защищают от двух ключевых ошибок: повторного попадания одного окна и неявного использования непригодных target/mask значений.

## 3. Feature groups, split protocol and preprocessing manifest

Feature groups собираются явно. `macro_window_sec` и `micro_window_sec` остаются metadata и не включаются в predictive features. KPI series order восстанавливается из `feature_schema.json`; если schema не содержит полного списка, fallback строится из имён колонок contract.


In [3]:
def extract_kpi_bases_from_columns(columns, stat_order=STAT_ORDER):
    suffixes = tuple('__' + stat for stat in stat_order)
    bases = []
    for col in columns:
        if col.startswith('kpi_') and col.endswith(suffixes):
            for stat in stat_order:
                suffix = '__' + stat
                if col.endswith(suffix):
                    bases.append(col[:-len(suffix)])
                    break
    return sorted(set(bases))

schema_bases = [item['feature_base'] for item in schema.get('kpi', {}).get('feature_bases', []) if isinstance(item, dict) and 'feature_base' in item]
column_bases = extract_kpi_bases_from_columns(df.columns)
if schema_bases:
    selected_series = [base for base in schema_bases if any((base + '__' + stat) in df.columns for stat in STAT_ORDER)]
    series_order_source = 'feature_schema.json'
else:
    selected_series = column_bases
    series_order_source = 'contract_column_names'

if not selected_series:
    raise ValueError('No KPI series bases could be recovered for tensorization')

missing_stat_columns = []
kpi_feature_cols = []
for base in selected_series:
    for stat in STAT_ORDER:
        col = base + '__' + stat
        if col in df.columns:
            kpi_feature_cols.append(col)
        else:
            missing_stat_columns.append(col)

leakage_cols = set(schema.get('excluded_from_predictive_features', [])) | {
    'workload', 'scenario', 'run_id', 'group_id', 'failure_family', 'scenario_type', 'state',
    'anchor_policy', 'window_ts', 'window_end', 'run_start', 'run_end', 'fail_start',
    'event_anchor', 'minutes_before_fail', 'target_available', 'target_policy',
    'split', 'split_group', 'y_warning', 'y_risk', 'y_anomaly', 'macro_window_sec', 'micro_window_sec'
}
predictive_feature_names = set(LOG_MACRO_COLS + LOG_MICRO_COLS + MASK_COLS + KPI_MICRO_COLS + kpi_feature_cols + ['log_text'])
leakage_intersection = sorted((predictive_feature_names - {'log_text'}) & leakage_cols)
if leakage_intersection:
    raise ValueError(f'Leakage columns entered predictive feature lists: {leakage_intersection}')

primary_train_idx = df.index[df['split'] == 'train'].to_numpy(dtype=np.int64)
test_idx = df.index[df['split'] == 'test'].to_numpy(dtype=np.int64)
if len(primary_train_idx) == 0 or len(test_idx) == 0:
    raise ValueError('Primary split must contain non-empty train and test parts')

def make_inner_val_split(frame, candidate_idx, group_col='run_id', val_size=0.20, seed=SEED):
    sub = frame.iloc[candidate_idx].reset_index(drop=True)
    groups = sub[group_col].astype(str).to_numpy()
    for offset in range(50):
        splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed + offset)
        inner_pos, val_pos = next(splitter.split(sub, groups=groups))
        inner_idx = candidate_idx[inner_pos]
        val_idx = candidate_idx[val_pos]
        warning_ok = frame.iloc[inner_idx]['y_warning'].nunique() == 2 and frame.iloc[val_idx]['y_warning'].nunique() == 2
        risk_ok = frame.iloc[inner_idx]['y_risk'].nunique() == 2 and frame.iloc[val_idx]['y_risk'].nunique() == 2
        if warning_ok and risk_ok:
            return inner_idx, val_idx, {'method': 'GroupShuffleSplit_by_run_id', 'seed': seed + offset, 'val_size': val_size, 'risk_classes_ok': risk_ok}
    return inner_idx, val_idx, {'method': 'GroupShuffleSplit_by_run_id', 'seed': seed + offset, 'val_size': val_size, 'risk_classes_ok': risk_ok, 'warning': 'class balance fallback used'}

train_idx, val_idx, inner_split_info = make_inner_val_split(df, primary_train_idx)

split_passport = {
    'primary_split_source': 'contract_split_column' if not split_fallback_used else 'split_report_fallback',
    'inner_validation': inner_split_info,
    'train_rows': int(len(train_idx)),
    'val_rows': int(len(val_idx)),
    'test_rows': int(len(test_idx)),
    'train_runs': int(df.iloc[train_idx]['run_id'].nunique()),
    'val_runs': int(df.iloc[val_idx]['run_id'].nunique()),
    'test_runs': int(df.iloc[test_idx]['run_id'].nunique()),
    'train_workloads': sorted(df.iloc[train_idx]['workload'].unique().tolist()),
    'val_workloads': sorted(df.iloc[val_idx]['workload'].unique().tolist()),
    'test_workloads': sorted(df.iloc[test_idx]['workload'].unique().tolist()),
}

feature_manifest = {
    'contract_path': str(CONTRACT_PATH.relative_to(ROOT)),
    'series_order_source': series_order_source,
    'series_selection_strategy': 'all_schema_series_available_in_contract',
    'selected_series_count': int(len(selected_series)),
    'selected_series': selected_series,
    'stat_order': STAT_ORDER,
    'kpi_feature_cols_count': int(len(kpi_feature_cols)),
    'kpi_feature_cols': kpi_feature_cols,
    'missing_stat_columns_count': int(len(missing_stat_columns)),
    'missing_stat_columns_sample': missing_stat_columns[:20],
    'log_macro_cols': LOG_MACRO_COLS,
    'log_micro_cols': LOG_MICRO_COLS,
    'mask_cols': MASK_COLS,
    'model_mask_cols': MODEL_MASK_COLS,
    'kpi_micro_cols': KPI_MICRO_COLS,
    'excluded_leakage_cols': sorted(leakage_cols),
    'leakage_intersection': leakage_intersection,
    'split_passport': split_passport,
}

display(pd.DataFrame([{
    'log_macro_dim': len(LOG_MACRO_COLS),
    'log_micro_dim': len(LOG_MICRO_COLS),
    'mask_dim': len(MODEL_MASK_COLS),
    'kpi_micro_dim': len(KPI_MICRO_COLS),
    'selected_kpi_series': len(selected_series),
    'kpi_feature_cols': len(kpi_feature_cols),
    'series_order_source': series_order_source,
}]))
display(pd.DataFrame([split_passport]).drop(columns=['train_workloads', 'val_workloads', 'test_workloads']))


,log_macro_dim,log_micro_dim,mask_dim,kpi_micro_dim,selected_kpi_series,kpi_feature_cols,series_order_source
0,6,7,4,4,432,2160,feature_schema.json


,primary_split_source,inner_validation,train_rows,val_rows,test_rows,train_runs,val_runs,test_runs
0,contract_split_column,"{'method': 'GroupShuffleSplit_by_run_id', 'see...",5949,1721,2567,261,66,113


### Локальный вывод
Feature manifest фиксирует не только список колонок, но и порядок series/stat в KPI tensor. Это важно: модель не должна зависеть от случайного порядка колонок CSV.

## 4. Preprocessing: HF log embeddings, numeric contexts and KPI tensor

Текстовая ветка строится через `SentenceTransformer` с моделью `google/embeddinggemma-300m`: тексты нормализуются, уникальные строки кодируются один раз и сохраняются в `.npz` cache. Статистические scalers для embedding/numeric/KPI fit-ятся только на inner-train rows.


In [4]:
def normalize_log_text(text, max_chars=600):
    value = str(text).strip()
    return value[:max_chars] if value else '__NO_LOG__'

def deterministic_hashing_embeddings(texts, dim=HASHING_FALLBACK_DIM):
    arr = np.zeros((len(texts), dim), dtype=np.float32)
    for i, text in enumerate(texts):
        for token in str(text).lower().split():
            h = int(hashlib.sha1(token.encode('utf-8')).hexdigest(), 16)
            arr[i, h % dim] += 1.0 if ((h >> 8) & 1) else -1.0
        norm = np.linalg.norm(arr[i]) + 1e-8
        arr[i] /= norm
    return arr.astype(np.float32)

def resolve_hf_token_for_encoder():
    for env_name in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGINGFACE_TOKEN']:
        token = os.getenv(env_name)
        if isinstance(token, str) and token.strip().startswith('hf_') and len(token.strip()) > 20:
            return token.strip(), f'env:{env_name}'
    try:
        from huggingface_hub import get_token
        token = get_token()
        if isinstance(token, str) and token.strip().startswith('hf_') and len(token.strip()) > 20:
            return token.strip(), 'huggingface_hub cache'
    except Exception:
        pass
    return None, 'missing'

def configure_hf_runtime():
    os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
    os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
    os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '240')
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '0')
    os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')

def load_google_sentence_encoder(enable_hf=USE_HF_GOOGLE_EMBEDDER, model_name=HF_GOOGLE_EMBEDDER_MODEL, allow_download=ALLOW_HF_DOWNLOAD):
    if not enable_hf:
        return None, 'hashing_fallback_disabled', HASHING_FALLBACK_DIM, {'hf_enabled': False}
    configure_hf_runtime()
    hf_token, hf_token_source = resolve_hf_token_for_encoder()
    manifest = {'hf_enabled': True, 'preferred_model': model_name, 'hf_token_available': hf_token is not None, 'hf_token_source': hf_token_source}
    try:
        from huggingface_hub import snapshot_download
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        manifest['import_error'] = str(exc)[:240]
        return None, 'hashing_fallback_import_error', HASHING_FALLBACK_DIM, manifest

    snapshot_path = None
    try:
        snapshot_path = snapshot_download(repo_id=model_name, token=hf_token, local_files_only=True)
        manifest['snapshot_source'] = 'local_hf_cache'
    except Exception as local_exc:
        manifest['local_cache_error'] = str(local_exc)[:240]
        if allow_download:
            try:
                snapshot_path = snapshot_download(repo_id=model_name, token=hf_token, local_files_only=False, resume_download=True, max_workers=2)
                manifest['snapshot_source'] = 'download_or_resumed_hf_cache'
            except Exception as dl_exc:
                manifest['download_error'] = str(dl_exc)[:300]

    try:
        load_target = snapshot_path if snapshot_path is not None else model_name
        encoder_device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = SentenceTransformer(load_target, device=encoder_device)
        dim = int(model.get_sentence_embedding_dimension())
        manifest.update({'encoder_device': encoder_device, 'embedding_dim_raw': dim, 'loaded_model': model_name, 'snapshot_available': snapshot_path is not None})
        return model, model_name, dim, manifest
    except Exception as exc:
        manifest['load_error'] = str(exc)[:300]
        return None, 'hashing_fallback_load_error', HASHING_FALLBACK_DIM, manifest

def embed_logs_with_hf_cache(texts, fit_idx, cache_dir=CACHE_DIR):
    raw_texts = pd.Series(texts).fillna('').astype(str)
    normalized = [normalize_log_text(t) for t in raw_texts.tolist()]
    empty_share = float(np.mean([t == '__NO_LOG__' for t in normalized]))
    hashes = [hashlib.sha1(t.encode('utf-8')).hexdigest() for t in normalized]
    encoder, encoder_name, embedding_dim, encoder_manifest = load_google_sentence_encoder()
    cache_name = encoder_name.replace('/', '__').replace(':', '-')
    cache_file = cache_dir / f'log_embeddings_{cache_name}.npz'

    cache_map = {}
    if cache_file.exists():
        try:
            payload = np.load(cache_file, allow_pickle=True)
            for h, e in zip(payload['hashes'], payload['embeddings']):
                arr = np.asarray(e, dtype=np.float32)
                if arr.shape[0] == embedding_dim:
                    cache_map[str(h)] = arr
        except Exception as exc:
            encoder_manifest['cache_read_warning'] = str(exc)[:220]

    unique_pairs = []
    seen = set()
    for h, t in zip(hashes, normalized):
        if h not in seen:
            unique_pairs.append((h, t))
            seen.add(h)
    missing_pairs = [(h, t) for h, t in unique_pairs if h not in cache_map]

    if missing_pairs:
        missing_texts = [t for _, t in missing_pairs]
        if encoder is not None:
            encoded = encoder.encode(missing_texts, batch_size=HF_EMBED_BATCH_SIZE, show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        else:
            encoded = deterministic_hashing_embeddings(missing_texts, dim=embedding_dim)
        for (h, _), e in zip(missing_pairs, encoded):
            cache_map[h] = np.asarray(e, dtype=np.float32)

    save_hashes = np.array(list(cache_map.keys()), dtype=object)
    save_embeddings = np.vstack([cache_map[h] for h in save_hashes]).astype(np.float32)
    np.savez_compressed(cache_file, hashes=save_hashes, embeddings=save_embeddings, model_name=encoder_name, dim=embedding_dim)

    raw_embeddings = np.vstack([cache_map[h] for h in hashes]).astype(np.float32)
    scaler = StandardScaler()
    scaler.fit(raw_embeddings[fit_idx])
    embeddings = scaler.transform(raw_embeddings).astype(np.float32)
    manifest = {
        'encoder_type': 'sentence_transformers_hf_google_embeddinggemma' if encoder is not None else 'deterministic_hashing_fallback',
        'encoder_name': encoder_name,
        'preferred_model': HF_GOOGLE_EMBEDDER_MODEL,
        'embedding_dim_raw': int(raw_embeddings.shape[1]),
        'embedding_dim': int(embeddings.shape[1]),
        'embedding_shape': list(embeddings.shape),
        'raw_embeddings_normalized_by_encoder': bool(encoder is not None),
        'cache_file': str(cache_file.relative_to(ROOT)),
        'unique_normalized_texts': int(len(unique_pairs)),
        'encoded_missing_texts_this_run': int(len(missing_pairs)),
        'empty_log_text_share': empty_share,
        'scaler_fit_rows': int(len(fit_idx)),
        'encoder_manifest': encoder_manifest,
    }
    return embeddings, {'encoder_name': encoder_name, 'scaler': scaler, 'cache_file': cache_file}, manifest

def fit_scaled_numeric_matrix(frame, cols, fit_idx, transform_fn=None):
    raw = frame[cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
    if transform_fn is not None:
        raw = transform_fn(raw.copy())
    x = raw.to_numpy(dtype=np.float32, copy=True)
    imputer = SimpleImputer(strategy='median')
    imputer.fit(x[fit_idx])
    x_imp = imputer.transform(x).astype(np.float32)
    scaler = StandardScaler()
    scaler.fit(x_imp[fit_idx])
    x_scaled = scaler.transform(x_imp).astype(np.float32)
    manifest = {
        'cols': cols,
        'shape': list(x_scaled.shape),
        'fit_rows': int(len(fit_idx)),
        'imputer_strategy': 'median',
        'scaler': 'StandardScaler',
    }
    return x_scaled, {'imputer': imputer, 'scaler': scaler}, manifest

def transform_kpi_micro(raw):
    raw['kpi_recency_sec'] = np.log1p(np.clip(pd.to_numeric(raw['kpi_recency_sec'], errors='coerce'), 0, None))
    return raw

def build_mask_matrix(frame, cols=MODEL_MASK_COLS):
    x = frame[cols].apply(pd.to_numeric, errors='coerce').fillna(0.0).to_numpy(dtype=np.float32)
    if x.shape[1] >= 3:
        x[:, :3] = np.clip(x[:, :3], 0.0, 1.0)
    if x.shape[1] >= 4:
        x[:, 3] = np.clip(x[:, 3], 0.0, 1.0)
    return x

def build_kpi_tensor(frame, selected_series, fit_idx, stat_order=STAT_ORDER):
    n = len(frame)
    s = len(selected_series)
    c_stats = len(stat_order)
    raw = np.empty((n, c_stats, s), dtype=np.float32)
    missing_cols = []
    for j, stat in enumerate(stat_order):
        cols = [base + '__' + stat for base in selected_series]
        existing = [col if col in frame.columns else None for col in cols]
        block = np.full((n, s), np.nan, dtype=np.float32)
        present_positions = [pos for pos, col in enumerate(existing) if col is not None]
        present_cols = [existing[pos] for pos in present_positions]
        if present_cols:
            block[:, present_positions] = frame[present_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float32, copy=True)
        missing_cols.extend([cols[pos] for pos, col in enumerate(existing) if col is None])
        raw[:, j, :] = block
    availability_channel = np.isfinite(raw).any(axis=1).astype(np.float32)
    train_raw = raw[fit_idx]
    with np.errstate(invalid='ignore', divide='ignore'):
        means = np.nanmean(train_raw, axis=0).astype(np.float32)
        stds = np.nanstd(train_raw, axis=0).astype(np.float32)
    means = np.where(np.isfinite(means), means, 0.0).astype(np.float32)
    stds = np.where(np.isfinite(stds) & (stds > 1e-6), stds, 1.0).astype(np.float32)
    normalized = (raw - means[None, :, :]) / stds[None, :, :]
    normalized = np.where(np.isfinite(normalized), normalized, 0.0).astype(np.float32)
    tensor = np.concatenate([normalized, availability_channel[:, None, :]], axis=1).astype(np.float32)
    manifest = {
        'shape': list(tensor.shape),
        'normalization': 'per_stat_per_series_standardization_fit_on_inner_train',
        'numeric_imputation_after_scaling': 0.0,
        'availability_channel_index': int(c_stats),
        'missing_stat_columns_count': int(len(missing_cols)),
        'fit_rows': int(len(fit_idx)),
    }
    stats = {'means': means, 'stds': stds}
    return tensor, availability_channel, stats, manifest

log_embedding, text_artifacts, text_manifest = embed_logs_with_hf_cache(df['log_text'], train_idx)
log_macro, macro_artifacts, macro_manifest = fit_scaled_numeric_matrix(df, LOG_MACRO_COLS, train_idx)
log_micro, micro_artifacts, micro_manifest = fit_scaled_numeric_matrix(df, LOG_MICRO_COLS, train_idx)
kpi_micro, kpi_micro_artifacts, kpi_micro_manifest = fit_scaled_numeric_matrix(df, KPI_MICRO_COLS, train_idx, transform_fn=transform_kpi_micro)
mask_matrix = build_mask_matrix(df)
kpi_tensor, kpi_availability_channel, kpi_norm_stats, kpi_tensor_manifest = build_kpi_tensor(df, selected_series, train_idx)

targets = {target: df[target].to_numpy(dtype=np.float32) for target in TARGET_COLS}
arrays = {
    'log_embedding': log_embedding.astype(np.float32),
    'log_macro': log_macro.astype(np.float32),
    'log_micro': log_micro.astype(np.float32),
    'kpi_tensor': kpi_tensor.astype(np.float32),
    'kpi_micro': kpi_micro.astype(np.float32),
    'masks': mask_matrix.astype(np.float32),
    'y_warning': targets['y_warning'],
    'y_risk': targets['y_risk'],
}

feature_manifest['tensor_shapes'] = {
    'log_embedding': list(arrays['log_embedding'].shape),
    'log_macro': list(arrays['log_macro'].shape),
    'log_micro': list(arrays['log_micro'].shape),
    'kpi_tensor': list(arrays['kpi_tensor'].shape),
    'kpi_micro': list(arrays['kpi_micro'].shape),
    'masks': list(arrays['masks'].shape),
}
feature_manifest['text_encoder'] = {'encoder_name': text_manifest.get('encoder_name'), 'embedding_dim': text_manifest.get('embedding_dim'), 'cache_file': text_manifest.get('cache_file')}

preprocessing_manifest = {
    'train_only_policy': 'pretrained HF text encoder is fixed; embedding scaler and all numeric/KPI statistics are fitted on inner train split only',
    'contract_integrity_report': integrity_report,
    'text_embedding': text_manifest,
    'log_macro_numeric': macro_manifest,
    'log_micro_numeric': micro_manifest,
    'kpi_micro_numeric': kpi_micro_manifest,
    'kpi_tensor': kpi_tensor_manifest,
    'mask_matrix': {'cols': MODEL_MASK_COLS, 'shape': list(mask_matrix.shape), 'scaling': 'raw clipped masks/ratio'},
}

shape_table = pd.DataFrame([
    {'input': 'log_embedding', 'shape': str(arrays['log_embedding'].shape)},
    {'input': 'log_macro', 'shape': str(arrays['log_macro'].shape)},
    {'input': 'log_micro', 'shape': str(arrays['log_micro'].shape)},
    {'input': 'kpi_tensor', 'shape': str(arrays['kpi_tensor'].shape)},
    {'input': 'kpi_micro', 'shape': str(arrays['kpi_micro'].shape)},
    {'input': 'masks', 'shape': str(arrays['masks'].shape)},
])
display(shape_table)
display(pd.DataFrame([text_manifest]))
gc.collect()


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

C:\Users\ipopi\AppData\Local\Temp\ipykernel_26516\3743074467.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = int(model.get_sentence_embedding_dimension())


,input,shape
0,log_embedding,"(10237, 768)"
1,log_macro,"(10237, 6)"
2,log_micro,"(10237, 7)"
3,kpi_tensor,"(10237, 6, 432)"
4,kpi_micro,"(10237, 4)"
5,masks,"(10237, 4)"


,encoder_type,encoder_name,preferred_model,embedding_dim_raw,embedding_dim,embedding_shape,raw_embeddings_normalized_by_encoder,cache_file,unique_normalized_texts,encoded_missing_texts_this_run,empty_log_text_share,scaler_fit_rows,encoder_manifest
0,sentence_transformers_hf_google_embeddinggemma,google/embeddinggemma-300m,google/embeddinggemma-300m,768,768,"[10237, 768]",True,outputs\raw_central_model\cache\log_embeddings...,3752,0,0.431865,5949,"{'hf_enabled': True, 'preferred_model': 'googl..."


6331

### Локальный вывод
Data inputs собраны в пять независимых блоков: HF log embedding, macro log context, micro log dynamics, series-aware KPI tensor и KPI/coverage numeric context. Pretrained text encoder не обучается на contract, а все статистические scalers fit-ятся только на inner-train rows.

## 5. Architecture card

| Block | Input | Output | Purpose |
|---|---|---|---|
| HF text encoder | `log_text` | `log_embedding` | fixed `google/embeddinggemma-300m` semantic vector |
| Log macro | `[log_embedding, log_macro]` | `h_macro` | semantic + 60s context |
| Log micro | `log_micro` | `h_micro` | 10s dynamics |
| Log fuse | `[h_macro, h_micro]` | `h_log` | log modality representation |
| KPI branch | `kpi_tensor + kpi_micro` | `h_kpi` | series-aware KPI dynamics |
| Gate | `[h_log, h_kpi, masks]` | scalar `g` | missing-aware late fusion |
| Dual heads | `h_fused` | warning/risk logits | primary + secondary targets |
| Auxiliary heads | `h_log`, `h_kpi`, `h_fused` | warning logits | stabilize missing-aware branches |

Gate convention: `g` near 1 means the fused representation is log-dominant; `g` near 0 means KPI-dominant. Fallback rules force log-only/KPI-only behavior when one modality is absent.


In [5]:
class CentralFusionDataset(Dataset):
    def __init__(self, arrays, indices):
        self.arrays = arrays
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return int(len(self.indices))

    def __getitem__(self, item):
        idx = self.indices[item]
        return {
            'log_embedding': torch.from_numpy(self.arrays['log_embedding'][idx]),
            'log_macro': torch.from_numpy(self.arrays['log_macro'][idx]),
            'log_micro': torch.from_numpy(self.arrays['log_micro'][idx]),
            'kpi_tensor': torch.from_numpy(self.arrays['kpi_tensor'][idx]),
            'kpi_micro': torch.from_numpy(self.arrays['kpi_micro'][idx]),
            'masks': torch.from_numpy(self.arrays['masks'][idx]),
            'y_warning': torch.tensor(self.arrays['y_warning'][idx], dtype=torch.float32),
            'y_risk': torch.tensor(self.arrays['y_risk'][idx], dtype=torch.float32),
            'row_index': torch.tensor(idx, dtype=torch.long),
        }

def move_batch_to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

class DenseBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class KPIBranch1DCNN(nn.Module):
    def __init__(self, in_channels, micro_dim, out_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.micro_proj = DenseBlock(micro_dim, hidden_dim=32, dropout=0.10)
        self.out_proj = nn.Sequential(nn.Linear(64, out_dim), nn.ReLU())

    def forward(self, kpi_tensor, kpi_micro):
        h_cnn = self.conv(kpi_tensor).flatten(1)
        h_micro = self.micro_proj(kpi_micro)
        return self.out_proj(torch.cat([h_cnn, h_micro], dim=1))

class CentralFusionMultiRate(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden = int(config['hidden_dim'])
        self.modality_dropout = float(np.clip(config.get('modality_dropout', 0.0), 0.0, 0.95))
        self.use_aux_heads = bool(config.get('use_aux_heads', True))
        self.log_macro_branch = DenseBlock(config['log_embedding_dim'] + config['log_macro_dim'], hidden_dim=hidden, dropout=0.15)
        self.log_micro_branch = DenseBlock(config['log_micro_dim'], hidden_dim=hidden, dropout=0.10)
        self.log_fuser = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU())
        self.kpi_branch = KPIBranch1DCNN(config['kpi_channels'], config['kpi_micro_dim'], out_dim=hidden)
        self.gate = nn.Linear(hidden * 2 + 3, 1)
        self.warning_head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
        self.risk_head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
        if self.use_aux_heads:
            self.log_only_head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))
            self.kpi_only_head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))
            self.fused_aux_head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))

    def apply_modality_dropout(self, log_mask, kpi_mask):
        if (not self.training) or self.modality_dropout <= 0:
            return log_mask, kpi_mask
        both = (log_mask > 0.5) & (kpi_mask > 0.5)
        rnd = torch.rand_like(log_mask)
        drop_log = both & (rnd < self.modality_dropout / 2.0)
        drop_kpi = both & (rnd >= self.modality_dropout / 2.0) & (rnd < self.modality_dropout)
        log_eff = torch.where(drop_log, torch.zeros_like(log_mask), log_mask)
        kpi_eff = torch.where(drop_kpi, torch.zeros_like(kpi_mask), kpi_mask)
        return log_eff, kpi_eff

    def compute_gate(self, log_h, kpi_h, log_mask, kpi_mask, kpi_ratio):
        gate_raw = torch.sigmoid(self.gate(torch.cat([log_h, kpi_h, log_mask, kpi_mask, kpi_ratio], dim=1)))
        both = (log_mask > 0.5) & (kpi_mask > 0.5)
        log_only = (log_mask > 0.5) & (kpi_mask <= 0.5)
        kpi_only = (log_mask <= 0.5) & (kpi_mask > 0.5)
        none = (log_mask <= 0.5) & (kpi_mask <= 0.5)
        gate = gate_raw
        gate = torch.where(log_only, torch.ones_like(gate), gate)
        gate = torch.where(kpi_only, torch.zeros_like(gate), gate)
        gate = torch.where(none, torch.full_like(gate, 0.5), gate)
        gate = torch.where(both, gate_raw, gate)
        return gate

    def forward(self, batch):
        masks = batch['masks']
        log_mask = masks[:, 0:1]
        kpi_mask = masks[:, 1:2]
        kpi_ratio = masks[:, 3:4] if masks.shape[1] > 3 else kpi_mask
        log_mask_eff, kpi_mask_eff = self.apply_modality_dropout(log_mask, kpi_mask)
        log_macro_h = self.log_macro_branch(torch.cat([batch['log_embedding'], batch['log_macro']], dim=1))
        log_micro_h = self.log_micro_branch(batch['log_micro'])
        log_h = self.log_fuser(torch.cat([log_macro_h, log_micro_h], dim=1))
        kpi_h = self.kpi_branch(batch['kpi_tensor'], batch['kpi_micro'])
        log_h = log_h * log_mask_eff
        kpi_h = kpi_h * kpi_mask_eff
        gate = self.compute_gate(log_h, kpi_h, log_mask_eff, kpi_mask_eff, kpi_ratio)
        fused = gate * log_h + (1.0 - gate) * kpi_h
        out = {
            'warning_logit': self.warning_head(fused).squeeze(1),
            'risk_logit': self.risk_head(fused).squeeze(1),
            'gate': gate.squeeze(1),
        }
        if self.use_aux_heads:
            out['aux_log_warning'] = self.log_only_head(log_h).squeeze(1)
            out['aux_kpi_warning'] = self.kpi_only_head(kpi_h).squeeze(1)
            out['aux_fused_warning'] = self.fused_aux_head(fused).squeeze(1)
        return out

def compute_pos_weight(y):
    y = np.asarray(y).astype(np.float32)
    pos = float((y == 1).sum())
    neg = float((y == 0).sum())
    if pos <= 0 or neg <= 0:
        return 1.0
    return neg / pos

def central_multitask_loss(outputs, batch, pos_weights, lambda_risk=LAMBDA_RISK, lambda_aux=LAMBDA_AUX):
    warning_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights['warning'])
    risk_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights['risk'])
    y_warning = batch['y_warning']
    y_risk = batch['y_risk']
    loss_warning = warning_loss_fn(outputs['warning_logit'], y_warning)
    loss_risk = risk_loss_fn(outputs['risk_logit'], y_risk)
    aux_terms = []
    for key in ['aux_log_warning', 'aux_kpi_warning', 'aux_fused_warning']:
        if key in outputs:
            aux_terms.append(warning_loss_fn(outputs[key], y_warning))
    loss_aux = torch.stack(aux_terms).mean() if aux_terms else torch.tensor(0.0, device=y_warning.device)
    total = loss_warning + lambda_risk * loss_risk + lambda_aux * loss_aux
    return total, {'warning': loss_warning.detach(), 'risk': loss_risk.detach(), 'aux': loss_aux.detach(), 'total': total.detach()}

def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. One-batch graph validation

Этот блок проверяет совместимость tensors и фактической архитектуры: один mini-batch, forward pass, multitask loss и backward pass.


In [6]:
model_config = {
    'model_name': 'central_fusion_multirate',
    'text_encoder': text_manifest.get('encoder_name'),
    'run_full_training_default': RUN_FULL_TRAINING,
    'log_embedding_dim': int(arrays['log_embedding'].shape[1]),
    'log_macro_dim': int(arrays['log_macro'].shape[1]),
    'log_micro_dim': int(arrays['log_micro'].shape[1]),
    'kpi_channels': int(arrays['kpi_tensor'].shape[1]),
    'kpi_series_count': int(arrays['kpi_tensor'].shape[2]),
    'kpi_micro_dim': int(arrays['kpi_micro'].shape[1]),
    'mask_dim': int(arrays['masks'].shape[1]),
    'hidden_dim': HIDDEN_DIM,
    'modality_dropout': MODALITY_DROPOUT,
    'use_aux_heads': USE_AUX_HEADS,
    'lambda_risk': LAMBDA_RISK,
    'lambda_aux': LAMBDA_AUX,
    'batch_size_full_training': BATCH_SIZE,
    'max_epochs_full_training': MAX_EPOCHS,
    'patience_full_training': PATIENCE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'seed': SEED,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = CentralFusionDataset(arrays, train_idx)
dry_loader = DataLoader(train_dataset, batch_size=min(DRY_RUN_BATCH_SIZE, len(train_dataset)), shuffle=False)
model = CentralFusionMultiRate(model_config).to(device)
model_parameter_count = count_parameters(model)
model_config['model_parameter_count'] = model_parameter_count

pos_weights = {
    'warning': torch.tensor(compute_pos_weight(arrays['y_warning'][train_idx]), dtype=torch.float32, device=device),
    'risk': torch.tensor(compute_pos_weight(arrays['y_risk'][train_idx]), dtype=torch.float32, device=device),
}

batch = move_batch_to_device(next(iter(dry_loader)), device)
model.train()
outputs = model(batch)
batch_size_seen = int(batch['y_warning'].shape[0])
shape_checks = {
    'warning_logit': tuple(outputs['warning_logit'].shape) == (batch_size_seen,),
    'risk_logit': tuple(outputs['risk_logit'].shape) == (batch_size_seen,),
    'gate': tuple(outputs['gate'].shape) == (batch_size_seen,),
}
if not all(shape_checks.values()):
    raise RuntimeError(f'One-batch shape check failed: {shape_checks}')
loss, loss_parts = central_multitask_loss(outputs, batch, pos_weights)
loss_finite = bool(torch.isfinite(loss).detach().cpu().item())
if not loss_finite:
    raise RuntimeError('One-batch loss is not finite')
model.zero_grad(set_to_none=True)
loss.backward()
grad_norms = [p.grad.detach().abs().sum().item() for p in model.parameters() if p.grad is not None]
gradients_ok = bool(len(grad_norms) > 0 and np.isfinite(grad_norms).all() and np.sum(grad_norms) > 0)
if not gradients_ok:
    raise RuntimeError('One-batch backward pass did not produce finite non-zero gradients')

dry_run_report = {
    'run_full_training': bool(RUN_FULL_TRAINING),
    'rows_total': int(len(df)),
    'train_rows': int(len(train_idx)),
    'val_rows': int(len(val_idx)),
    'test_rows': int(len(test_idx)),
    'embedding_shape': list(arrays['log_embedding'].shape),
    'kpi_tensor_shape': list(arrays['kpi_tensor'].shape),
    'kpi_micro_shape': list(arrays['kpi_micro'].shape),
    'mask_shape': list(arrays['masks'].shape),
    'model_parameter_count': int(model_parameter_count),
    'forward_ok': True,
    'shape_checks': shape_checks,
    'loss_finite': loss_finite,
    'loss_value': float(loss.detach().cpu().item()),
    'loss_parts': {k: float(v.cpu().item()) for k, v in loss_parts.items()},
    'gradients_ok': gradients_ok,
    'validation_batch_size': batch_size_seen,
}

write_json_safe(OUT_ROOT / 'feature_manifest.json', feature_manifest)
write_json_safe(OUT_ROOT / 'preprocessing_manifest.json', preprocessing_manifest)
write_json_safe(OUT_ROOT / 'model_config.json', model_config)
write_json_safe(OUT_ROOT / 'dry_run_report.json', dry_run_report)

display(pd.DataFrame([dry_run_report]).drop(columns=['shape_checks', 'loss_parts']))
print('Saved validation artifacts:', [p.name for p in sorted(OUT_ROOT.glob('*.json'))])


,run_full_training,rows_total,train_rows,val_rows,test_rows,embedding_shape,kpi_tensor_shape,kpi_micro_shape,mask_shape,model_parameter_count,forward_ok,loss_finite,loss_value,gradients_ok,validation_batch_size
0,False,10237,5949,1721,2567,"[10237, 768]","[10237, 6, 432]","[10237, 4]","[10237, 4]",145161,True,True,2.112484,True,64


Saved validation artifacts: ['dry_run_report.json', 'feature_manifest.json', 'model_config.json', 'preprocessing_manifest.json']


### Локальный вывод
One-batch validation проверяет тот же computational graph, который используется в training protocol: HF embeddings, две логовые ветки, KPI 1D-CNN, gate, dual heads и auxiliary warning heads.

## 7. Full training, evaluation, ablation and inference store

Код ниже содержит полный protocol обучения, threshold selection, evaluation, ablation и inference store. Execution mode управляется флагом `RUN_FULL_TRAINING`.


In [7]:
def safe_pr_auc(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return None
    return float(average_precision_score(y_true, score))

def safe_roc_auc(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, score))

def binary_metrics_at_threshold(y_true, score, threshold):
    pred = (np.asarray(score) >= threshold).astype(int)
    y_true = np.asarray(y_true).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    return {'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'threshold': float(threshold)}

def select_threshold_on_val(y_true, score):
    if len(np.unique(y_true)) < 2:
        return 0.5, {'reason': 'validation target has one class'}
    thresholds = np.linspace(0.05, 0.95, 181)
    f1_values = [f1_score(y_true, np.asarray(score) >= thr, zero_division=0) for thr in thresholds]
    best = int(np.argmax(f1_values))
    return float(thresholds[best]), {'best_val_f1': float(f1_values[best])}

@torch.no_grad()
def predict_central_model(model, dataset, batch_size=BATCH_SIZE, device=device, ablation=None):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    rows = []
    for batch in loader:
        if ablation == 'force_no_kpi':
            batch['kpi_tensor'] = torch.zeros_like(batch['kpi_tensor'])
            batch['kpi_micro'] = torch.zeros_like(batch['kpi_micro'])
            batch['masks'] = batch['masks'].clone()
            batch['masks'][:, 1] = 0.0
            if batch['masks'].shape[1] > 2:
                batch['masks'][:, 2] = 0.0
        elif ablation == 'force_no_log':
            batch['log_embedding'] = torch.zeros_like(batch['log_embedding'])
            batch['log_macro'] = torch.zeros_like(batch['log_macro'])
            batch['log_micro'] = torch.zeros_like(batch['log_micro'])
            batch['masks'] = batch['masks'].clone()
            batch['masks'][:, 0] = 0.0
            if batch['masks'].shape[1] > 2:
                batch['masks'][:, 2] = 0.0
        batch = move_batch_to_device(batch, device)
        out = model(batch)
        rows.append(pd.DataFrame({
            'row_index': batch['row_index'].detach().cpu().numpy(),
            'warning_score': torch.sigmoid(out['warning_logit']).detach().cpu().numpy(),
            'risk_score': torch.sigmoid(out['risk_logit']).detach().cpu().numpy(),
            'fusion_gate': out['gate'].detach().cpu().numpy(),
            'y_warning': batch['y_warning'].detach().cpu().numpy(),
            'y_risk': batch['y_risk'].detach().cpu().numpy(),
        }))
    return pd.concat(rows, ignore_index=True)

def train_central_model(model, train_dataset, val_dataset, pos_weights, device=device):
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_state = None
    best_ap = -np.inf
    best_epoch = -1
    patience_left = PATIENCE
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = move_batch_to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)
            out = model(batch)
            loss, _ = central_multitask_loss(out, batch, pos_weights)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu().item()))
        val_pred = predict_central_model(model, val_dataset, batch_size=BATCH_SIZE, device=device)
        val_ap_warning = safe_pr_auc(val_pred['y_warning'], val_pred['warning_score'])
        val_ap_risk = safe_pr_auc(val_pred['y_risk'], val_pred['risk_score'])
        monitor = -np.inf if val_ap_warning is None else val_ap_warning
        row = {'epoch': epoch, 'train_loss': float(np.mean(train_losses)), 'val_pr_warning': val_ap_warning, 'val_pr_risk': val_ap_risk, 'lr': LEARNING_RATE}
        history.append(row)
        if monitor > best_ap + 1e-5:
            best_ap = monitor
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
        if patience_left <= 0:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return {'model': model, 'best_state': best_state, 'best_epoch': best_epoch, 'history': pd.DataFrame(history)}

def make_inference_table(pred_df, source_df, threshold_warning=0.5, threshold_risk=0.5):
    meta_cols = ['run_id', 'window_ts', 'split', 'workload', 'scenario', 'scenario_type', 'failure_family', 'anchor_policy', 'log_available', 'kpi_available', 'has_both_modalities', 'kpi_avail_ratio']
    meta = source_df.iloc[pred_df['row_index'].astype(int).to_numpy()][meta_cols].reset_index(drop=True)
    out = pd.concat([meta, pred_df[['y_warning', 'y_risk', 'warning_score', 'risk_score', 'fusion_gate']].reset_index(drop=True)], axis=1)
    out['threshold_warning'] = threshold_warning
    out['threshold_risk'] = threshold_risk
    out['pred_warning'] = (out['warning_score'] >= threshold_warning).astype(int)
    out['pred_risk'] = (out['risk_score'] >= threshold_risk).astype(int)
    out['pred_state'] = np.where(out['pred_risk'] == 1, 'risk', np.where(out['pred_warning'] == 1, 'warning', 'normal'))
    return out

def evaluate_prediction_table(table, target, score_col, threshold, slice_name='all'):
    y = table[target].astype(int).to_numpy()
    score = table[score_col].to_numpy()
    out = {
        'target': target,
        'slice': slice_name,
        'n': int(len(table)),
        'positives': int(y.sum()),
        'prevalence': float(y.mean()) if len(y) else None,
        'average_precision': safe_pr_auc(y, score),
        'roc_auc': safe_roc_auc(y, score),
    }
    selected_metrics = binary_metrics_at_threshold(y, score, threshold)
    half_metrics = binary_metrics_at_threshold(y, score, 0.5)
    out.update(selected_metrics)
    out.update({f'{k}_at_0_5': v for k, v in half_metrics.items() if k != 'threshold'})
    return out

def evaluate_by_slice(inference_df, threshold_warning, threshold_risk):
    slice_specs = [('all', np.ones(len(inference_df), dtype=bool))]
    slice_specs.append(('kpi_available=1', inference_df['kpi_available'].astype(int).to_numpy() == 1))
    slice_specs.append(('has_both_modalities=1', inference_df['has_both_modalities'].astype(int).to_numpy() == 1))
    for policy in sorted(inference_df['anchor_policy'].dropna().unique()):
        slice_specs.append((f'anchor_policy={policy}', inference_df['anchor_policy'].astype(str).to_numpy() == str(policy)))
    rows = []
    for name, mask in slice_specs:
        part = inference_df.loc[mask].copy()
        if len(part) == 0:
            continue
        rows.append(evaluate_prediction_table(part, 'y_warning', 'warning_score', threshold_warning, name))
        rows.append(evaluate_prediction_table(part, 'y_risk', 'risk_score', threshold_risk, name))
    return pd.DataFrame(rows)

def build_baseline_comparison(baseline_summary, central_metrics):
    baseline_rows = baseline_summary.get('baseline_metrics', [])
    keep = []
    for row in baseline_rows:
        if row.get('baseline') in ['availability-only', 'log-only', 'kpi-only', 'fused'] and row.get('slice') in ['all', 'has_both_modalities=1']:
            keep.append(row)
    central = central_metrics[central_metrics['slice'].isin(['all', 'has_both_modalities=1'])].copy()
    central['baseline'] = 'central-fusion-model'
    central['status'] = 'ok'
    cols = sorted(set(pd.DataFrame(keep).columns).union(central.columns))
    return pd.concat([pd.DataFrame(keep), central], ignore_index=True, sort=False)[cols]

def run_modality_ablation(model, test_dataset, source_df, threshold_warning, threshold_risk):
    rows = []
    for ablation_name, ablation_key in [('normal', None), ('force_no_kpi', 'force_no_kpi'), ('force_no_log', 'force_no_log')]:
        pred = predict_central_model(model, test_dataset, device=device, ablation=ablation_key)
        table = make_inference_table(pred, source_df, threshold_warning, threshold_risk)
        metrics = evaluate_by_slice(table, threshold_warning, threshold_risk)
        metrics['ablation'] = ablation_name
        rows.append(metrics)
    return pd.concat(rows, ignore_index=True)

def build_gate_analysis(inference_df):
    df_gate = inference_df.copy()
    df_gate['kpi_avail_ratio_bucket'] = pd.cut(df_gate['kpi_avail_ratio'], bins=[-0.01, 0.25, 0.5, 0.75, 1.0], labels=['0-0.25', '0.25-0.5', '0.5-0.75', '0.75-1.0'])
    rows = []
    for group_cols in [['has_both_modalities'], ['anchor_policy'], ['kpi_avail_ratio_bucket'], ['y_warning'], ['y_risk']]:
        grouped = df_gate.groupby(group_cols, dropna=False)['fusion_gate'].agg(['count', 'mean', 'std']).reset_index()
        grouped['grouping'] = '+'.join(group_cols)
        rows.append(grouped)
    return pd.concat(rows, ignore_index=True, sort=False)

def select_interpretability_cases(inference_df, source_df, kpi_tensor, selected_series, top_n=6):
    table = inference_df.copy()
    table['case_type'] = 'unselected'
    table.loc[(table['y_warning'] == 1) & (table['pred_warning'] == 1), 'case_type'] = 'true_positive_warning'
    table.loc[(table['y_warning'] == 0) & (table['pred_warning'] == 1), 'case_type'] = 'false_positive_warning'
    table.loc[(table['y_warning'] == 1) & (table['pred_warning'] == 0), 'case_type'] = 'false_negative_warning'
    selected = []
    for case_type in ['true_positive_warning', 'false_positive_warning', 'false_negative_warning']:
        part = table[table['case_type'] == case_type].sort_values('warning_score', ascending=False)
        if len(part):
            selected.append(part.iloc[0])
    if len(selected) < top_n:
        extra = table.sort_values('risk_score', ascending=False).head(top_n - len(selected))
        selected.extend([row for _, row in extra.iterrows()])
    rows = []
    seen = set()
    for row in selected[:top_n]:
        source_pos = int(row['row_index']) if 'row_index' in row.index else None
        key = (row['run_id'], row['window_ts'])
        if key in seen or source_pos is None:
            continue
        seen.add(key)
        abs_series = np.abs(kpi_tensor[source_pos, :len(STAT_ORDER), :]).max(axis=0)
        top_pos = np.argsort(abs_series)[-5:][::-1]
        top_drivers = [{'series': selected_series[i], 'abs_normalized_value': float(abs_series[i])} for i in top_pos]
        rows.append({
            'run_id': row['run_id'],
            'window_ts': row['window_ts'],
            'scenario_type': row['scenario_type'],
            'failure_family': row['failure_family'],
            'anchor_policy': row['anchor_policy'],
            'case_type': row['case_type'],
            'y_warning': int(row['y_warning']),
            'y_risk': int(row['y_risk']),
            'warning_score': float(row['warning_score']),
            'risk_score': float(row['risk_score']),
            'pred_state': row['pred_state'],
            'fusion_gate': float(row['fusion_gate']),
            'log_available': int(row['log_available']),
            'kpi_available': int(row['kpi_available']),
            'has_both_modalities': int(row['has_both_modalities']),
            'kpi_avail_ratio': float(row['kpi_avail_ratio']),
            'log_excerpt': str(source_df.iloc[source_pos]['log_text'])[:500],
            'top_kpi_drivers_json': json.dumps(top_drivers, ensure_ascii=False),
            'driver_note': 'descriptive top absolute normalized KPI values, not causal attribution',
        })
    return pd.DataFrame(rows)

def lookup_prediction(run_id, window_ts=None, top_n=5, path=OUT_ROOT / 'inference_store.csv'):
    store = pd.read_csv(path)
    part = store[store['run_id'].astype(str) == str(run_id)].copy()
    if window_ts is not None:
        part = part[part['window_ts'].astype(str) == str(window_ts)]
    return part.sort_values(['warning_score', 'risk_score'], ascending=False).head(top_n)

if RUN_FULL_TRAINING:
    set_seed(SEED)
    train_dataset_full = CentralFusionDataset(arrays, train_idx)
    val_dataset = CentralFusionDataset(arrays, val_idx)
    test_dataset = CentralFusionDataset(arrays, test_idx)
    model_full = CentralFusionMultiRate(model_config).to(device)
    pos_weights_full = {
        'warning': torch.tensor(compute_pos_weight(arrays['y_warning'][train_idx]), dtype=torch.float32, device=device),
        'risk': torch.tensor(compute_pos_weight(arrays['y_risk'][train_idx]), dtype=torch.float32, device=device),
    }
    training_result = train_central_model(model_full, train_dataset_full, val_dataset, pos_weights_full, device=device)
    history_df = training_result['history']
    history_df.to_csv(OUT_ROOT / 'training_history.csv', index=False)
    val_pred = predict_central_model(training_result['model'], val_dataset, device=device)
    threshold_warning, warning_thr_info = select_threshold_on_val(val_pred['y_warning'], val_pred['warning_score'])
    threshold_risk, risk_thr_info = select_threshold_on_val(val_pred['y_risk'], val_pred['risk_score'])
    test_pred = predict_central_model(training_result['model'], test_dataset, device=device)
    inference_df = make_inference_table(test_pred, df, threshold_warning, threshold_risk)
    inference_df['row_index'] = test_pred['row_index'].astype(int).to_numpy()
    inference_df.to_csv(OUT_ROOT / 'inference_store.csv', index=False)
    inference_df.to_csv(OUT_ROOT / 'central_model_predictions.csv', index=False)
    metrics_df = evaluate_by_slice(inference_df, threshold_warning, threshold_risk)
    metrics_df.to_csv(OUT_ROOT / 'coverage_metrics.csv', index=False)
    comparison_df = build_baseline_comparison(baseline_summary, metrics_df)
    comparison_df.to_csv(OUT_ROOT / 'baseline_comparison.csv', index=False)
    ablation_df = run_modality_ablation(training_result['model'], test_dataset, df, threshold_warning, threshold_risk)
    ablation_df.to_csv(OUT_ROOT / 'ablation_metrics.csv', index=False)
    gate_df = build_gate_analysis(inference_df)
    gate_df.to_csv(OUT_ROOT / 'gate_analysis.csv', index=False)
    cases_df = select_interpretability_cases(inference_df, df, arrays['kpi_tensor'], selected_series)
    cases_df.to_csv(OUT_ROOT / 'interpretability_cases.csv', index=False)
    torch.save({'state_dict': training_result['best_state'], 'model_config': model_config, 'feature_manifest': feature_manifest, 'threshold_warning': threshold_warning, 'threshold_risk': threshold_risk}, OUT_ROOT / 'central_model.pt')
    if joblib is not None:
        joblib.dump({'text': text_artifacts, 'macro': macro_artifacts, 'micro': micro_artifacts, 'kpi_micro': kpi_micro_artifacts, 'kpi_norm_stats': kpi_norm_stats, 'preprocessing_manifest': preprocessing_manifest}, OUT_ROOT / 'preprocessing_artifacts.joblib')
    metrics_payload = {
        'run_full_training': True,
        'best_epoch': int(training_result['best_epoch']),
        'threshold_warning': threshold_warning,
        'threshold_risk': threshold_risk,
        'threshold_selection': {'warning': warning_thr_info, 'risk': risk_thr_info},
        'primary_metrics': metrics_df.to_dict(orient='records'),
        'warnings': ['Proxy anchors are evaluated as contract labels, not exact fault onset timestamps'],
    }
    write_json_safe(OUT_ROOT / 'central_model_metrics.json', metrics_payload)
    split_repeated = pd.DataFrame([{'mode': 'primary_split', 'reason': 'primary workload holdout from notebook 02 is the comparison contract'}])
    split_repeated.to_csv(OUT_ROOT / 'split_repeated_metrics.csv', index=False)
    model_card = f'''# Model card: central fusion model

Intended use: research prototype for AIOps warning/risk scoring on the raw multi-rate contract.

Data: {len(df)} windows, {df['run_id'].nunique()} runs, {df['workload'].nunique()} workloads. Targets are `y_warning` and `y_risk`; proxy anchors remain a limitation and must not be read as exact fault onset.

Architecture: log macro branch, log micro branch, KPI 1D-CNN branch, missing-aware gated fusion, dual heads.

Training protocol: group-aware inner validation, early stopping by validation PR-AUC for `y_warning`, modality dropout={MODALITY_DROPOUT}, loss weights risk={LAMBDA_RISK}, aux={LAMBDA_AUX}.

Primary metric: PR-AUC for `y_warning`; secondary metric: PR-AUC for `y_risk`. See `central_model_metrics.json`, `coverage_metrics.csv` and `baseline_comparison.csv`.

Limitations: proxy anchors, coverage shortcut, high-dimensional KPI tensor, small workload-holdout test, no heavy attribution beyond descriptive KPI drivers.
'''
    (OUT_ROOT / 'model_card_central_fusion.md').write_text(model_card, encoding='utf-8')
    display(metrics_df.head(20))


## 8. Artifact contract and final interpretation

Этот блок проверяет artifact contract: feature manifest, preprocessing manifest, one-batch validation report и model config. При полном запуске рядом сохраняются training/evaluation artifacts.


In [8]:
from IPython.display import Markdown

expected_dry = ['feature_manifest.json', 'preprocessing_manifest.json', 'dry_run_report.json', 'model_config.json']
artifact_status = {name: (OUT_ROOT / name).exists() for name in expected_dry}
if not all(artifact_status.values()):
    raise RuntimeError(f'Missing expected validation artifacts: {artifact_status}')

training_artifacts = ['central_model_metrics.json', 'central_model_predictions.csv', 'central_model.pt', 'training_history.csv', 'model_card_central_fusion.md', 'coverage_metrics.csv', 'ablation_metrics.csv', 'gate_analysis.csv', 'interpretability_cases.csv', 'inference_store.csv', 'preprocessing_artifacts.joblib', 'baseline_comparison.csv', 'split_repeated_metrics.csv']
training_artifact_status = {name: (OUT_ROOT / name).exists() for name in training_artifacts}

for json_name in expected_dry + (['central_model_metrics.json'] if (OUT_ROOT / 'central_model_metrics.json').exists() else []):
    with (OUT_ROOT / json_name).open('r', encoding='utf-8') as f:
        json.load(f)

final_md = f'''## Финальный вывод для ВКР

- **Data contract:** `{CONTRACT_PATH.relative_to(ROOT)}`; rows={len(df):,}, runs={df['run_id'].nunique()}, workloads={df['workload'].nunique()}.
- **Text encoder:** `{text_manifest.get('encoder_name')}`; embedding shape `{arrays['log_embedding'].shape}`; cache `{text_manifest.get('cache_file')}`.
- **Inputs prepared:** log macro `{arrays['log_macro'].shape}`, log micro `{arrays['log_micro'].shape}`, KPI tensor `{arrays['kpi_tensor'].shape}`, KPI micro `{arrays['kpi_micro'].shape}`, masks `{arrays['masks'].shape}`.
- **Split sizes:** train={len(train_idx):,}, validation={len(val_idx):,}, test={len(test_idx):,}. Primary test workloads: `{', '.join(split_passport['test_workloads'])}`.
- **Graph validation:** forward={dry_run_report['forward_ok']}, loss_finite={dry_run_report['loss_finite']}, gradients_ok={dry_run_report['gradients_ok']}, parameters={dry_run_report['model_parameter_count']:,}.
- **Saved artifacts:** `{', '.join(expected_dry)}` in `{OUT_ROOT.relative_to(ROOT)}`.

### Архитектура
1. `google/embeddinggemma-300m` encodes normalized `log_text`; embeddings are standardized on inner-train rows.
2. Log macro branch: `DenseBlock([embedding, log_macro] -> 128 -> 64)` captures semantic context and 60s log aggregates.
3. Log micro branch: `DenseBlock(log_micro -> 128 -> 64)` captures 10s log dynamics.
4. Log fuser: `Linear(128 -> 64)` combines macro and micro log representations.
5. KPI branch: `Conv1d(C=6 -> 32) -> Conv1d(32 -> 32) -> AdaptiveMaxPool1d`, concatenated with `DenseBlock(kpi_micro -> 32)`, then projected to `h_kpi=64`.
6. Gate: `sigmoid(Linear([h_log, h_kpi, log_mask, kpi_mask, kpi_ratio] -> 1))` with hard fallback rules for missing modalities.
7. Heads: `warning_head` and `risk_head` are `64 -> 32 -> 1`; auxiliary warning heads use `64 -> 16 -> 1` for log-only, KPI-only and fused representations.

Локальный вывод: центральная gated-fusion architecture и все входные tensors подготовлены и проверены на одном mini-batch с фактическим HF Google embedder.
'''
display(Markdown(final_md))
display(pd.DataFrame([artifact_status]))
if RUN_FULL_TRAINING:
    display(pd.DataFrame([training_artifact_status]))


## Финальный вывод для ВКР

- **Data contract:** `outputs\raw_multirate\aligned_multirate_windows.csv`; rows=10,237, runs=440, workloads=16.
- **Text encoder:** `google/embeddinggemma-300m`; embedding shape `(10237, 768)`; cache `outputs\raw_central_model\cache\log_embeddings_google__embeddinggemma-300m.npz`.
- **Inputs prepared:** log macro `(10237, 6)`, log micro `(10237, 7)`, KPI tensor `(10237, 6, 432)`, KPI micro `(10237, 4)`, masks `(10237, 4)`.
- **Split sizes:** train=5,949, validation=1,721, test=2,567. Primary test workloads: `aggregation, join, rf, svd`.
- **Graph validation:** forward=True, loss_finite=True, gradients_ok=True, parameters=145,161.
- **Saved artifacts:** `feature_manifest.json, preprocessing_manifest.json, dry_run_report.json, model_config.json` in `outputs\raw_central_model`.

### Архитектура
1. `google/embeddinggemma-300m` encodes normalized `log_text`; embeddings are standardized on inner-train rows.
2. Log macro branch: `DenseBlock([embedding, log_macro] -> 128 -> 64)` captures semantic context and 60s log aggregates.
3. Log micro branch: `DenseBlock(log_micro -> 128 -> 64)` captures 10s log dynamics.
4. Log fuser: `Linear(128 -> 64)` combines macro and micro log representations.
5. KPI branch: `Conv1d(C=6 -> 32) -> Conv1d(32 -> 32) -> AdaptiveMaxPool1d`, concatenated with `DenseBlock(kpi_micro -> 32)`, then projected to `h_kpi=64`.
6. Gate: `sigmoid(Linear([h_log, h_kpi, log_mask, kpi_mask, kpi_ratio] -> 1))` with hard fallback rules for missing modalities.
7. Heads: `warning_head` and `risk_head` are `64 -> 32 -> 1`; auxiliary warning heads use `64 -> 16 -> 1` for log-only, KPI-only and fused representations.

Локальный вывод: центральная gated-fusion architecture и все входные tensors подготовлены и проверены на одном mini-batch с фактическим HF Google embedder.


,feature_manifest.json,preprocessing_manifest.json,dry_run_report.json,model_config.json
0,True,True,True,True
